# 10 — Workability: IS baseline + residual GP

**What this is.** The slump model, rebuilt so the Indian Standard supplies the water
effect and the Gaussian process learns only what the standard does not know.

**Why it matters here.** In this dataset water never moves on its own — it moves opposite
to superplasticiser (r = −0.79) and binder (r = −0.80). A model fitted on it cannot
recover which way water pushes slump, and the one currently in the repository pushes it
the wrong way.

**What breaks if it is ignored.** The optimiser asks for more slump and gets told to take
water *out*, so every candidate it proposes is wrong in the one dimension the plant cares
about most.

---

## The finding this notebook is built on

This is not an assertion. Cell 3 below re-derives all of it from
`04_workability_dataset.csv` every time the notebook is run.

| what | value |
|---|---|
| rows | 264 usable |
| design groups | 12 — of which only 5 are real families (M20–M40-P-20MS) |
| slump variance **between** groups | ≈ 80 percent |
| water ↔ slump | r ≈ −0.09 |
| water ↔ superplasticiser | r ≈ −0.79 |
| water ↔ binder | r ≈ −0.80 |
| water span **inside** a group | 5.1 to 5.8 kg |

At the IS slope a 5.4 kg span is worth about 24 mm of slump, against an observed
within-group span of 37 to 48 mm. **There is no observation in which water moved while
superplasticiser and binder were held.** The water coefficient is therefore
*unidentifiable* — not mis-fitted.

## What follows from that

`[PC]` Water enters the slump prediction **only** through the IS 10262 relation. The
residual model is refused every feature that is a function of batch water. Gate **G1** —
the water direction is positive everywhere — then holds **by construction**, not as a test
that might fail.

`[PC]` The residual model **must** carry superplasticiser dose. SP is the strongest
legitimate correlate of slump here and the thing water is confounded with; leaving it out
puts the confound straight back in.

## Classification

| tag | applies to |
|---|---|
| `[IS]` | Table 4, cl 5.3, the 3 percent per 25 mm — supplied by `is_rules` |
| `[EI]` | inverting cl 5.3 into a slump-response slope. cl 5.3 is a **design** rule and ends *"or may be established by trial"*. **IS 10262 states no slump-response slope anywhere.** This must never be written up as an IS requirement. |
| `[ML]` | the residual GP, its kernel, its features |
| `[PC]` | the feature exclusion, the group quarantine, the validation protocol |

In [1]:
# EXPORT: skip -- notebook preamble only
# Always load the CURRENT project modules from disk. Python imports a module once per
# kernel, so a kernel that was open before an earlier notebook rewrote layer0.py /
# screen.py / ... would silently keep the OLD code. Dropping them first makes every
# Run All pick up the latest files, with or without a kernel restart.
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import is_rules as IS
import layer0 as L0

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

DATA = "04_workability_dataset.csv"
raw = pd.read_csv(DATA)
print(f"{DATA}: {len(raw)} rows, {raw.shape[1]} columns")

is_rules loaded.
source verification: all 9 standards OK
04_workability_dataset.csv: 299 rows, 116 columns


## 1. The diagnostics, re-derived

Gate for this section: the numbers printed here must match the table in the header. If they
drift, the dataset changed and every conclusion below has to be re-read.

In [2]:
# EXPORT: skip -- diagnostics only
SLUMP, WATER, MSA = "slump_mm", "water_kg_m3", "msa"
GRP = "design_grp"

d = raw.dropna(subset=[SLUMP, WATER, MSA]).copy()
d[GRP] = d[GRP].fillna(d.get("design_id"))
print(f"usable rows            {len(d)}")
print(f"design groups          {d[GRP].nunique()}")

# variance decomposition -- how much signal is between mixes vs inside one mix
gmean = d.groupby(GRP)[SLUMP].transform("mean")
ss_between = ((gmean - d[SLUMP].mean()) ** 2).sum()
ss_within = ((d[SLUMP] - gmean) ** 2).sum()
frac_between = 100 * ss_between / (ss_between + ss_within)
print(f"slump variance BETWEEN groups {frac_between:5.1f} %   WITHIN {100-frac_between:5.1f} %")
print(f"-> the effective sample size is the number of GROUPS, not the number of rows")

print("\ncorrelation of measured slump with")
for c in ("water_kg_m3", "sp_pct_binder", "sp_kg_m3", "binder_kg_m3",
          "wb_ratio", "fine_frac_of_agg", "scm_frac", "concrete_temp"):
    if c in d:
        print(f"  {c:20s} r = {d[c].corr(d[SLUMP]):+.3f}")

print("\ncorrelation of WATER with")
for c in ("sp_pct_binder", "sp_kg_m3", "binder_kg_m3", "wb_ratio"):
    if c in d:
        print(f"  {c:20s} r = {d[c].corr(d[WATER]):+.3f}")

print("\nwater span inside each design group (kg/m3)")
span = d.groupby(GRP).agg(n=(SLUMP, "size"),
                          water_span=(WATER, lambda s: s.max() - s.min()),
                          slump_span=(SLUMP, lambda s: s.max() - s.min()))
print(span.sort_values("n", ascending=False).to_string())

slope = 1.0 / (IS.IS10262_TABLE4_WATER[20] * IS.IS10262_CL_5_3_SLUMP_PCT_PER_25MM / 100.0 / 25.0)
print(f"\n[EI] IS 10262 cl 5.3 slope at 20 mm msa: {slope:.2f} mm of slump per kg of water")
print(f"     a {span['water_span'].max():.1f} kg span is worth about "
      f"{span['water_span'].max()*slope:.0f} mm, against observed within-group spans of "
      f"{span['slump_span'].min():.0f}-{span['slump_span'].max():.0f} mm")

usable rows            264
design groups          12
slump variance BETWEEN groups  80.1 %   WITHIN  19.9 %
-> the effective sample size is the number of GROUPS, not the number of rows

correlation of measured slump with
  water_kg_m3          r = -0.090
  sp_pct_binder        r = +0.522
  sp_kg_m3             r = +0.480
  binder_kg_m3         r = +0.349
  wb_ratio             r = -0.254
  fine_frac_of_agg     r = +0.523
  scm_frac             r = +0.796
  concrete_temp        r = -0.009

correlation of WATER with
  sp_pct_binder        r = -0.743
  sp_kg_m3             r = -0.793
  binder_kg_m3         r = -0.795
  wb_ratio             r = +0.850

water span inside each design group (kg/m3)
                                          n  water_span  slump_span
design_grp                                                         
M30-P-20MS-01                            82        5.76        48.0
M25-P-20MS-01                            64        5.79        43.0
M35-P-20MS-02              

## 2. Quarantine — groups that are not design families

`[PC]` Seven of the twelve group keys are numeric concatenations
(`150.0|110.0|0.0|170.`), each carrying one or two rows. They are not design families; they
are rows whose design identity was lost upstream. Fitting on them inflates the apparent
group count without adding an experiment.

The project's statistical-integrity rule requires the **true** number of independent
experimental conditions to be reported. Quarantining these is how that number becomes
honest. They are kept in a separate frame, not deleted.

In [3]:
def is_design_family(key) -> bool:
    """[PC] A real design family has a design ID, not a numeric concatenation.

    Keys like '150.0|110.0|0.0|170.' are rows whose design identity was lost
    upstream; keys like 'M30-P-20MS-01' are design families. The test is whether
    the key contains a letter.
    """
    return isinstance(key, str) and any(ch.isalpha() for ch in key)


def split_quarantine(df, group_col="design_grp", min_rows=3):
    """Separate real design families from orphans. Returns (kept, quarantined, report)."""
    counts = df[group_col].value_counts()
    keep_keys = [k for k, n in counts.items() if is_design_family(k) and n >= min_rows]
    kept = df[df[group_col].isin(keep_keys)].copy()
    quar = df[~df[group_col].isin(keep_keys)].copy()
    report = {
        "n_rows_in": len(df),
        "n_rows_kept": len(kept),
        "n_rows_quarantined": len(quar),
        "groups_in": int(df[group_col].nunique()),
        "groups_kept": len(keep_keys),
        "kept_keys": sorted(keep_keys),
        "quarantined_keys": sorted(map(str, quar[group_col].unique())),
        "rule": f"[PC] keep groups with an alphabetic design ID and at least {min_rows} rows",
    }
    return kept, quar, report

In [4]:
# EXPORT: skip -- reporting only
kept, quar, qrep = split_quarantine(d)
print(f"rows      {qrep['n_rows_in']} -> kept {qrep['n_rows_kept']}, quarantined {qrep['n_rows_quarantined']}")
print(f"groups    {qrep['groups_in']} -> kept {qrep['groups_kept']}")
print(f"\nkept (these are the independent experimental conditions):")
for k in qrep["kept_keys"]:
    print(f"   {k:24s} n={int((kept[GRP]==k).sum())}")
print(f"\nquarantined: {qrep['quarantined_keys']}")
print(f"\nTRUE NUMBER OF INDEPENDENT EXPERIMENTAL CONDITIONS: {qrep['groups_kept']}")
print("This is the number that goes in the report. Not 264.")

rows      264 -> kept 256, quarantined 8
groups    12 -> kept 5

kept (these are the independent experimental conditions):
   M20-P-20MS-01            n=31
   M25-P-20MS-01            n=64
   M30-P-20MS-01            n=82
   M35-P-20MS-02            n=51
   M40-P-20MS-02            n=28

quarantined: ['150.0|110.0|0.0|170.0|564.0|463.0|952.0', '200.0|100.0|0.0|170.0|555.0|456.0|937.0', '215.0|0.0|215.0|169.0|555.0|444.0|818.0', '310.0|100.0|0.0|170.0|564.0|470.0|816.0', '340.0|100.0|0.0|173.0|545.0|454.0|786.0', '340.0|100.0|0.0|173.0|554.0|461.0|801.0', '355.0|85.0|0.0|173.0|547.0|456.0|789.0']

TRUE NUMBER OF INDEPENDENT EXPERIMENTAL CONDITIONS: 5
This is the number that goes in the report. Not 264.


## 3. Fine-fraction compliance on the training rows

`[IS]` IS 10262 Table 5 with cl 5.5.2 fully applied caps the fine fraction at 0.460 at
20 mm even in Zone I. Table 5 Note 3 pushes the permitted fine fraction **down** for
crushed sand, not up, so it does not license anything higher.

**Correction.** An earlier version reported that none of the five families exceeded this
cap. That compared the dataset's `fine_frac_of_agg` — a **mass** fraction — against a
Table 5 limit that is a **volume** fraction. Recomputed by volume, with each row's own
specific gravities, **9 rows, all in M40, exceed even the most permissive cap**. Both
numbers are printed below so the claim stays checkable.

Rows above the cap are flagged, not dropped and not silently fitted. A flag is a question
for the plant, not a verdict.

In [5]:
# EXPORT: skip -- reporting only
FF = "fine_frac_of_agg"


def fine_volume_fraction(df):
    """Fine aggregate as a VOLUME fraction of total aggregate, from masses and each
    row's own specific gravities.

    IS 10262 Table 5 is a VOLUME table. The dataset column fine_frac_of_agg is a MASS
    fraction (fine_kg / (fine_kg + coarse_kg)). Comparing the two -- which an earlier
    version of this notebook did -- reports a mass fraction against a volume limit.
    With sand SG ~2.65 and coarse SG ~2.74 the volume fraction runs about 0.008 higher.
    """
    ssg = df["sand_sg"].fillna(2.65) if "sand_sg" in df else 2.65
    csg = df["ca_sg"].fillna(2.74) if "ca_sg" in df else 2.74
    vf = df["fine_agg_kg_m3"] / ssg
    vc = df["coarse_agg_kg_m3"] / csg
    return vf / (vf + vc)


kept = kept.copy()
kept["fine_vol_frac"] = fine_volume_fraction(kept)
quar = quar.copy()
if len(quar) and "fine_agg_kg_m3" in quar:
    quar["fine_vol_frac"] = fine_volume_fraction(quar)

# the most permissive case the standard allows at 20 mm: Zone I, w/c 0.50, cl 5.5.2 applied
ca_min = IS.coarse_aggregate_volume("M30", 20, "I", 0.50, pumpable=True)["ca_volume_fraction"]
cap = 1.0 - ca_min
over = kept[kept["fine_vol_frac"] > cap + 1e-9]
print(f"fine fraction, MASS (dataset column)  {kept[FF].min():.4f} .. {kept[FF].max():.4f}")
print(f"fine fraction, VOLUME (Table 5 basis) {kept['fine_vol_frac'].min():.4f} .. "
      f"{kept['fine_vol_frac'].max():.4f}")
print(f"[IS] most the code permits            {cap:.4f} by VOLUME"
      f"   (Table 5 Zone I @ 20 mm, w/c 0.50, cl 5.5.2 -10%)")
print(f"rows above that cap, by volume        {len(over)} of {len(kept)}"
      f"   ({100*len(over)/max(len(kept),1):.1f} %)")
print(f"rows above it if MASS were (wrongly) used: "
      f"{int((kept[FF] > cap + 1e-9).sum())}   <- what an earlier version reported")
if len(over):
    print("\n  affected families:")
    for k, n in over[GRP].value_counts().items():
        print(f"     {k:24s} {n:3d} rows, max fine VOLUME fraction "
              f"{over.loc[over[GRP]==k, 'fine_vol_frac'].max():.4f}")
    print("\n  This is the MOST permissive cap. [IS] cl 5.5.1 raises the coarse volume -- and")
    print("  so LOWERS the permitted fine fraction -- by 0.01 for every 0.05 the w/c falls")
    print("  below 0.50, so each row's own cap is tighter than this. [U] A compliance")
    print("  QUESTION for the plant, not a verdict; the rows stay in the fit, flagged.")

fine fraction, MASS (dataset column)  0.3888 .. 0.4597
fine fraction, VOLUME (Table 5 basis) 0.3949 .. 0.4679
[IS] most the code permits            0.4600 by VOLUME   (Table 5 Zone I @ 20 mm, w/c 0.50, cl 5.5.2 -10%)
rows above that cap, by volume        9 of 256   (3.5 %)
rows above it if MASS were (wrongly) used: 0   <- what an earlier version reported

  affected families:
     M40-P-20MS-02              9 rows, max fine VOLUME fraction 0.4679

  This is the MOST permissive cap. [IS] cl 5.5.1 raises the coarse volume -- and
  so LOWERS the permitted fine fraction -- by 0.01 for every 0.05 the w/c falls
  below 0.50, so each row's own cap is tighter than this. [U] A compliance
  QUESTION for the plant, not a verdict; the rows stay in the fit, flagged.


## 4. The IS baseline and the residual target

`[IS]` The baseline is `is_rules.water_content` inverted. No constant is written here.

```
W(S) = W_table4(msa) x (1 + 0.03 x (S - 50) / 25)      IS 10262 Table 4 + cl 5.3
S(W) = 50 + (W / W_table4(msa) - 1) x (25 / 0.03)      the inverse   [EI]
```

`[IS]` cl 5.3 carries **two** provisions and they must not be collapsed:

**(a)** water demand rises with target slump — about 3 percent per 25 mm on the Table 4
base. This is the slump–water relation.
**(b)** the water so calculated **may then be reduced** by chemical admixture — 5 to 10
percent for water reducers, 20 to 30 percent and above for superplasticizers.

Provision **(b)** is taken *in order to hold the same slump*. Annex A-6 computes 191.58 kg
for 75 mm and then batches far less because a superplasticizer is used — it is still a
75 mm mix. **So the slump a batch implies comes from its gross water, not its net water.**
The net water is un-reduced first:

```
W_gross = W_net / (1 - reduction/100)
S       = 50 + (W_gross / W_table4 - 1) x (25 / 0.03)
```

Inverting the **net** water directly — which an earlier version of this notebook did —
says that adding superplasticizer *lowers* slump, because it lowers water. That is
backwards, and it was visible downstream: predicted slump fell from 103 mm to 34 mm as the
dose rose. With the correction the baseline is **centred** on the measurements (mean 109.4
against a measured 109.5) and the residual has mean ≈ 0.

The reduction must come from a source independent of the measured slump — the fitted
dose relation in §4b — or the baseline is circular.

In [6]:
def _table4_base(msa_mm, ca_shape="angular"):
    # [IS] Table 4 row + cl 5.3 shape adjustment, elementwise. Refuses unlisted rows.
    adj = IS.IS10262_CL_5_3_SHAPE_ADJUST_KG[ca_shape]
    rows = IS.IS10262_TABLE4_WATER

    def one(m):
        m = float(m)
        key = m if m in rows else (int(m) if int(m) == m and int(m) in rows else None)
        if key is None:
            raise IS.MissingTableRow(
                f"IS 10262 Table 4 prints no row for msa {m} mm. "
                f"Printed rows: {sorted(rows)}. Do not interpolate."
            )
        return rows[key] + adj

    arr = np.asarray(msa_mm)
    if arr.ndim == 0:
        return float(one(arr.item()))
    return np.array([one(v) for v in arr.ravel()], dtype=float).reshape(arr.shape)


def is_slump_baseline(water_kg_m3, msa_mm, ca_shape="angular",
                      water_reduction_pct=0.0):
    """[EI] The IS-implied slump for a batch. Mean function for the GP.

    IS 10262 cl 5.3 has TWO separate provisions and they must not be collapsed:

      (a) water demand rises with target slump -- about 3 percent per 25 mm, on the
          Table 4 base. This is the slump-water relation.
      (b) the water so calculated MAY THEN BE REDUCED by chemical admixture, 5-10
          percent for water reducers and 20-30 percent and above for superplasticizers.

    Provision (b) is taken *in order to hold the same slump*. Annex A-6 computes 191.58
    kg for 75 mm and then batches far less because a superplasticizer is used -- the
    mix is still a 75 mm mix. So the slump a batch implies is the slump that generated
    its GROSS water, not the slump you would read off its net water.

    Inverting the net water directly -- which an earlier version of this notebook did --
    says that adding superplasticizer LOWERS slump, because it lowers water. That is
    backwards, and it was visible in the screen: predicted slump fell from 103 to 34 mm
    as the dose rose.

    So the net water is un-reduced first:

        W_gross = W_net / (1 - reduction/100)
        S       = 50 + (W_gross / W_table4 - 1) x (25 / 0.03)

    `water_reduction_pct` must come from a source independent of the measured slump --
    `water_reduction_pct(sp_dose, fit)` from the dose -- or the baseline is circular.

    Still NOT a predicted slump: it carries no plant effect, no aggregate shape beyond
    the cl 5.3 offset, no temperature. The residual model supplies those.
    """
    base = _table4_base(msa_mm, ca_shape)
    pct = IS.IS10262_CL_5_3_SLUMP_PCT_PER_25MM / 100.0
    w_net = np.asarray(water_kg_m3, dtype=float)
    red = np.asarray(water_reduction_pct, dtype=float) / 100.0
    if np.any(red >= 1.0):
        raise ValueError("water reduction of 100 percent or more is not physical")
    w_gross = w_net / (1.0 - red)
    return 50.0 + (w_gross / base - 1.0) * (25.0 / pct)


def is_slump_slope(msa_mm, ca_shape="angular"):
    """[EI] dS/dW in mm of slump per kg of water. Constant for a given msa."""
    base = _table4_base(msa_mm, ca_shape)
    return (25.0 / (IS.IS10262_CL_5_3_SLUMP_PCT_PER_25MM / 100.0)) / base


# [PC] Anything that is a function of batch water. The residual model may not see these,
#      or it can re-learn -- and re-break -- the direction the baseline exists to fix.
WATER_DERIVED = frozenset({
    "water_kg_m3", "water_batched", "free_water", "wb_ratio", "wc_ratio", "paste_frac",
})

# [PC] What the residual IS allowed to see. SP dose is mandatory: it is the thing water
#      is confounded with, and the strongest legitimate correlate of slump in this data.
RESIDUAL_FEATURES = [
    "sp_pct_binder",        # [IS] IS 9103 cl 10.2 bounds it; the model learns inside that
    "binder_kg_m3",
    "scm_frac",
    "fine_frac_of_agg",     # determined by IS 10262 Table 5 in Layer 0, but it VARIES by zone
    "msand_frac",
    "agg_binder_ratio",
    "concrete_temp",        # [IS] cl 5.3 names temperature as a water-demand driver
]


def check_no_water_leak(features):
    """Gate G1, enforced at build time rather than hoped for at test time."""
    leak = sorted(set(features) & WATER_DERIVED)
    if leak:
        raise ValueError(
            f"features {leak} are functions of batch water. Water may enter the slump "
            f"prediction only through is_slump_baseline(), or gate G1 is void."
        )
    return True


check_no_water_leak(RESIDUAL_FEATURES)

True

## 4b. The cl 5.3 admixture water reduction, fitted from this plant

`[IS]` IS 10262 cl 5.3 permits the calculated water to be **reduced** by chemical
admixture — 5 to 10 percent for water reducers, 20 to 30 percent and above for
superplasticizers — but fixes no value. Annex C-6 took **23 percent** "based on trial data".

Layer 0 therefore cannot assume a number. It must be supplied. Borrowing Annex C's 23
percent would be borrowing another example's trial result, so it is **fitted from the five
design families instead**: the reduction each family actually used is recoverable, because
the Table 4 + cl 5.3 demand for its *measured* slump is computable and the batched water is
recorded.

This is the one relation in this notebook that generalises across families.

In [7]:
def implied_water_reduction_pct(water_kg_m3, slump_mm, msa_mm, ca_shape="angular"):
    """[V] The cl 5.3 reduction a batch actually used, backed out of the record.

        reduction = 1 - (water batched) / (Table 4 + cl 5.3 demand for the MEASURED slump)

    This is an OBSERVATION, not a standard value: cl 5.3 gives ranges and an explicit
    "or may be established by trial". This is that trial, run on the plant's own history.
    """
    demand = np.array([
        IS.water_content("M30", float(m), float(s), ca_shape=ca_shape)["water_kg_m3"]
        for m, s in zip(np.atleast_1d(msa_mm), np.atleast_1d(slump_mm))])
    return 100.0 * (1.0 - np.atleast_1d(np.asarray(water_kg_m3, float)) / demand)


def fit_water_reduction(df, *, water_col="water_kg_m3", slump_col="slump_mm",
                        msa_col="msa", sp_col="sp_pct_binder"):
    """[V] Fit reduction% against superplasticiser dose. Straight line, no tuning.

    Returns the coefficients plus a leave-one-family-out error, so the number that
    goes into Layer 0 carries its own out-of-family accuracy.
    """
    d = df.dropna(subset=[water_col, slump_col, msa_col, sp_col]).copy()
    y = implied_water_reduction_pct(d[water_col], d[slump_col], d[msa_col])
    x = d[sp_col].to_numpy(float)
    b, a = np.polyfit(x, y, 1)
    pred = a + b * x
    r2 = 1.0 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)

    errs, per_family = [], []
    if "design_grp" in d:
        for g in sorted(d["design_grp"].unique()):
            tr, te = d[d["design_grp"] != g], d[d["design_grp"] == g]
            if len(tr) < 5 or len(te) < 2:
                continue
            ytr = implied_water_reduction_pct(tr[water_col], tr[slump_col], tr[msa_col])
            bb, aa = np.polyfit(tr[sp_col].to_numpy(float), ytr, 1)
            yte = implied_water_reduction_pct(te[water_col], te[slump_col], te[msa_col])
            e = (aa + bb * te[sp_col].to_numpy(float)) - yte
            errs.extend(e.tolist())
            per_family.append({"group": g, "n": len(te), "bias_pp": float(np.mean(e)),
                               "mae_pp": float(np.mean(np.abs(e)))})

    return {"intercept_pct": float(a), "slope_pct_per_sp_pct": float(b),
            "r2": float(r2), "n": len(d),
            # the ranges this relation was FITTED on. Outside them it is extrapolation,
            # and it sets the water for every candidate -- so it must be guarded.
            "sp_pct_range": (float(d[sp_col].min()), float(d[sp_col].max())),
            "water_kg_m3_range": (float(d[water_col].min()), float(d[water_col].max())),
            "slump_mm_range": (float(d[slump_col].min()), float(d[slump_col].max())),
            "logo_mae_pp": float(np.mean(np.abs(errs))) if errs else None,
            "per_family": per_family,
            "basis": ("[V] fitted from this plant's own families; IS 10262 cl 5.3 permits "
                      "the reduction but states no value")}


def water_reduction_pct(sp_pct_of_binder, fit):
    """[V] The cl 5.3 reduction to hand Layer 0 for a given admixture dose."""
    return fit["intercept_pct"] + fit["slope_pct_per_sp_pct"] * np.asarray(
        sp_pct_of_binder, dtype=float)


def inside_fitted_range(fit, *, sp_pct=None, water_kg_m3=None, slump_mm=None):
    """[PC] Is a candidate inside the range the cl 5.3 relation was fitted on?

    The support box in notebook 11 covers the GP's seven features. Water is
    deliberately NOT one of them -- that is what makes gate G1 hold by construction --
    so nothing there guards water. But the reduction relation and the baseline were
    both fitted inside a narrow water band, and the relation now SETS the water for
    every candidate. Outside that band both are extrapolating, unguarded.

    Returns a per-axis verdict. True means inside; None means not checked.
    """
    out = {}
    for name, val, key in (("sp_pct_binder", sp_pct, "sp_pct_range"),
                           ("water_kg_m3", water_kg_m3, "water_kg_m3_range"),
                           ("slump_mm", slump_mm, "slump_mm_range")):
        if val is None or key not in fit:
            out[name] = None
            continue
        lo, hi = fit[key]
        v = np.asarray(val, dtype=float)
        out[name] = (v >= lo) & (v <= hi)
        out[name + "_range"] = (lo, hi)
    return out

In [8]:
# EXPORT: skip -- reporting only
WFIT = fit_water_reduction(kept)
print(f"reduction% = {WFIT['intercept_pct']:.3f} + {WFIT['slope_pct_per_sp_pct']:.3f}"
      f" x SP%_of_binder     R2 {WFIT['r2']:.4f}   n {WFIT['n']}")
print(f"leave-one-family-out MAE {WFIT['logo_mae_pp']:.2f} percentage points"
      f"  (about {WFIT['logo_mae_pp']/100*200:.1f} kg of water)")
print()
for r in WFIT["per_family"]:
    print(f"   {r['group']:16s} n={r['n']:3d}  bias {r['bias_pp']:+6.2f} pp   MAE {r['mae_pp']:5.2f} pp")
print()
print("[IS] cl 5.3 quotes 5-10% for water reducers and 20-30% and above for")
print("     superplasticizers. This plant sits at", end=" ")
_lo, _hi = water_reduction_pct(kept['sp_pct_binder'].min(), WFIT), water_reduction_pct(kept['sp_pct_binder'].max(), WFIT)
print(f"{float(_lo):.1f}-{float(_hi):.1f}% over its dose range,")
print("     which is BETWEEN the two quoted bands. [U] Whether the admixture in use is a")
print("     water reducer, a mid-range or a superplasticizer below its appropriate dosage")
print("     is a material question for the plant, not something this fit can settle.")

reduction% = 3.709 + 14.037 x SP%_of_binder     R2 0.7303   n 256
leave-one-family-out MAE 1.41 percentage points  (about 2.8 kg of water)

   M20-P-20MS-01    n= 31  bias  +0.78 pp   MAE  1.15 pp
   M25-P-20MS-01    n= 64  bias  -0.17 pp   MAE  1.36 pp
   M30-P-20MS-01    n= 82  bias  +0.72 pp   MAE  1.44 pp
   M35-P-20MS-02    n= 51  bias  -0.80 pp   MAE  1.46 pp
   M40-P-20MS-02    n= 28  bias  -1.47 pp   MAE  1.65 pp

[IS] cl 5.3 quotes 5-10% for water reducers and 20-30% and above for
     superplasticizers. This plant sits at 12.4-24.4% over its dose range,
     which is BETWEEN the two quoted bands. [U] Whether the admixture in use is a
     water reducer, a mid-range or a superplasticizer below its appropriate dosage
     is a material question for the plant, not something this fit can settle.


In [9]:
# EXPORT: skip -- reporting only
kept = kept.copy()
kept["water_reduction_pct"] = water_reduction_pct(kept["sp_pct_binder"].to_numpy(float), WFIT)
kept["is_baseline_mm"] = is_slump_baseline(kept[WATER], kept[MSA],
                                           water_reduction_pct=kept["water_reduction_pct"])
kept["residual_mm"] = kept[SLUMP] - kept["is_baseline_mm"]

print(f"[EI] IS slope at 20 mm : {is_slump_slope(20):.3f} mm of slump per kg of water")
print()
for name, col in (("measured slump", SLUMP), ("IS baseline", "is_baseline_mm"),
                  ("residual", "residual_mm")):
    s = kept[col]
    print(f"  {name:16s} n={len(s):4d}  mean {s.mean():8.2f}  sd {s.std():7.2f}"
          f"  min {s.min():8.2f}  max {s.max():8.2f}")
print()
print("The baseline is now CENTRED on the measurements, because the net water is")
print("un-reduced by the fitted cl 5.3 relation before cl 5.3's slump-water term is")
print("inverted. An earlier version inverted the NET water, which made the baseline sit")
print("about 155 mm low and said superplasticiser LOWERS slump. The residual is now a")
print("residual: mean near zero, and it carries plant effects rather than a units error.")

[EI] IS slope at 20 mm : 4.480 mm of slump per kg of water

  measured slump   n= 256  mean   109.54  sd   13.08  min    75.00  max   142.00
  IS baseline      n= 256  mean   109.37  sd   14.34  min    77.95  max   148.14
  residual         n= 256  mean     0.16  sd   17.87  min   -47.23  max    51.98

The baseline is now CENTRED on the measurements, because the net water is
un-reduced by the fitted cl 5.3 relation before cl 5.3's slump-water term is
inverted. An earlier version inverted the NET water, which made the baseline sit
about 155 mm low and said superplasticiser LOWERS slump. The residual is now a
residual: mean near zero, and it carries plant effects rather than a units error.


## 5. Fit — residual GP, grouped

`[ML]` A Gaussian process on the residual. `[PC]` Validation is **leave-one-group-out**:
every cube from one physical mix goes to the same fold. Splitting them across folds would
let the model see a mix's own siblings in training, which is the leakage the project rules
forbid and is what makes a model look far better than it is.

With five families, leave-one-group-out is close to leave-one-point-out. Its error bar will
be wide. That is the honest picture, and the reason notebook 11 opens with an achievability
check rather than a promise.

In [10]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler


def build_gp(seed=0):
    """[ML] Residual GP. WhiteKernel is the MEASUREMENT noise term -- see predict()."""
    k = (ConstantKernel(1.0, (1e-3, 1e4))
         * RBF(length_scale=np.ones(1), length_scale_bounds=(1e-2, 1e3))
         + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-6, 1e4)))
    return GaussianProcessRegressor(kernel=k, normalize_y=True,
                                    n_restarts_optimizer=5, random_state=seed)


def fit_residual_model(df, features=None, *, slump_col="slump_mm",
                       water_col="water_kg_m3", msa_col="msa", seed=0,
                       water_fit=None, sp_col="sp_pct_binder"):
    """Fit the residual GP. Returns a dict carrying everything predict() needs.

    water_fit is the cl 5.3 reduction relation from fit_water_reduction(). It is
    REQUIRED: without it the baseline inverts net water and the admixture dose comes
    out with the wrong sign on slump.
    """
    if water_fit is None:
        raise ValueError(
            "fit_residual_model needs water_fit from fit_water_reduction(). Without the "
            "cl 5.3 reduction the baseline treats a water saving as a slump loss, which "
            "inverts the admixture effect.")
    features = list(features or RESIDUAL_FEATURES)
    features = [f for f in features if f in df.columns]
    check_no_water_leak(features)

    sub = df.dropna(subset=features + [slump_col, water_col, msa_col, sp_col]).copy()
    red = water_reduction_pct(sub[sp_col].to_numpy(float), water_fit)
    base = is_slump_baseline(sub[water_col], sub[msa_col], water_reduction_pct=red)
    y = (sub[slump_col] - base).to_numpy(float)

    scaler = StandardScaler().fit(sub[features].to_numpy(float))
    X = scaler.transform(sub[features].to_numpy(float))

    kern = (ConstantKernel(1.0, (1e-3, 1e4))
            * RBF(length_scale=np.ones(X.shape[1]), length_scale_bounds=(1e-2, 1e3))
            + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-6, 1e4)))
    gp = GaussianProcessRegressor(kernel=kern, normalize_y=True,
                                  n_restarts_optimizer=5, random_state=seed)
    gp.fit(X, y)

    # The WhiteKernel noise_level lives in the NORMALISED y space, while
    # gp.predict(return_std=True) returns a standard deviation in ORIGINAL units.
    # Comparing them directly is a units error -- exactly the class of mistake the old
    # optimiser made -- so rescale it here, once, and store it in original units.
    y_std = float(getattr(gp, "_y_train_std", 1.0))
    noise = float(gp.kernel_.k2.noise_level) * (y_std ** 2)
    return {"gp": gp, "scaler": scaler, "features": features, "noise_var": noise,
            "water_fit": water_fit, "sp_col": sp_col,
            "noise_in_normalised_units": False,     # already rescaled above -- do not rescale again
            "y_train_std": y_std,
            "feature_median": {f: float(np.nanmedian(sub[f].to_numpy(float))) for f in features},
            "n": len(sub), "kernel": str(gp.kernel_),
            "trained_on_groups": sorted(map(str, sub.get("design_grp", pd.Series(dtype=str)).unique()))}


def predict_slump(model, df, *, water_col="water_kg_m3", msa_col="msa", kind="mean"):
    """Predicted slump = IS baseline + residual GP.

    kind='mean'     -> uncertainty of the MEAN slump of a mix. The WhiteKernel
                       measurement-noise term is REMOVED, because it describes the
                       scatter of individual readings about that mean.
    kind='specimen' -> uncertainty of ONE slump reading. The noise term is kept.

    These are different questions and IS 456 treats them differently (cl 15.1.1 and
    cl 16.1 for strength). Adding the noise term on top of a GP sd that already
    contains it -- as the previous optimiser did -- inflates every interval and
    makes the feasibility test reject good mixes.
    """
    if kind not in ("mean", "specimen"):
        raise ValueError("kind must be 'mean' or 'specimen'")
    X = model["scaler"].transform(df[model["features"]].to_numpy(float))
    mu_res, sd_full = model["gp"].predict(X, return_std=True)
    var = np.maximum(sd_full ** 2, 0.0)
    if kind == "mean":
        var = np.maximum(var - model["noise_var"], 1e-12)   # REMOVE, never add
    red = water_reduction_pct(df[model.get("sp_col", "sp_pct_binder")].to_numpy(float),
                              model["water_fit"])
    base = is_slump_baseline(df[water_col], df[msa_col], water_reduction_pct=red)
    return np.asarray(base) + mu_res, np.sqrt(var)


def logo_error(residuals_mm, per_group_bias_mm):
    """[PC] Out-of-group error, split into the two parts that behave differently.

    The GP's own sd is an IN-SAMPLE quantity -- about 2.7 mm here, against a
    leave-one-group-out error that reaches 40 mm. Using the in-sample sd would make
    the system announce a band it cannot hold.

    But the out-of-group error is NOT scatter. Most of it is a per-family systematic
    OFFSET: for two of five families |bias| equals MAE exactly, meaning every held-out
    row errs the same way. Treating an offset as the sigma of a symmetric normal
    centred on mu understates how wrong a single family can be -- measured coverage of
    the nominal 90 percent band was 43 percent for one family.

    So both are returned and both must be used: `rmse_mm` for the width, `bias_rms_mm`
    for how far the centre itself can sit from the truth.
    """
    r = np.asarray(residuals_mm, dtype=float)
    b = np.asarray(per_group_bias_mm, dtype=float)
    return {
        "rmse_mm": float(np.sqrt(np.mean(r ** 2))),
        "sd_mm": float(np.std(r, ddof=1)),
        "bias_rms_mm": float(np.sqrt(np.mean(b ** 2))),
        "bias_max_mm": float(np.max(np.abs(b))),
        "note": ("rmse_mm is the width; bias_rms_mm is how far the CENTRE can sit from "
                 "the truth for an unseen family. Quote both, never rmse alone."),
    }


def pooled_skill(logo_df):
    """[PC] Pooled 1 - SUM(SS_res)/SUM(SS_base), not the mean of per-group ratios.

    With group sums of squares spanning 7682 to 11767, the mean of five ratios is
    dominated by the single worst fold and is not the quantity anyone means by skill.
    """
    return float(1.0 - logo_df["ss_res"].sum() / logo_df["ss_base"].sum())

In [11]:
# EXPORT: skip -- fit and report
model = fit_residual_model(kept, water_fit=WFIT)
print(f"fitted on {model['n']} rows, {len(model['trained_on_groups'])} groups")
print(f"features  {model['features']}")
print(f"kernel    {model['kernel']}")
print(f"noise var {model['noise_var']:.4f} mm^2  (sd {math.sqrt(model['noise_var']):.2f} mm,"
      f" rescaled from normalised units by y_std={model['y_train_std']:.3f})")

mu_mean, sd_mean = predict_slump(model, kept, kind="mean")
mu_spec, sd_spec = predict_slump(model, kept, kind="specimen")
resid = kept[SLUMP].to_numpy(float) - mu_mean
print(f"\nin-sample (optimistic, do NOT report this):")
print(f"  MAE {np.abs(resid).mean():6.2f} mm   RMSE {np.sqrt((resid**2).mean()):6.2f} mm")
print(f"  mean sd, mean-of-mix question     {sd_mean.mean():6.2f} mm")
print(f"  mean sd, single-reading question  {sd_spec.mean():6.2f} mm")
print(f"  ratio {sd_spec.mean()/max(sd_mean.mean(),1e-9):.2f}x -- this is the gap the old")
print(f"  optimiser collapsed by ADDING the noise term instead of removing it")

fitted on 256 rows, 5 groups
features  ['sp_pct_binder', 'binder_kg_m3', 'scm_frac', 'fine_frac_of_agg', 'msand_frac', 'agg_binder_ratio', 'concrete_temp']
kernel    0.751**2 * RBF(length_scale=[0.298, 0.934, 0.878, 1e+03, 1, 792, 1e+03]) + WhiteKernel(noise_level=0.515)
noise var 163.9854 mm^2  (sd 12.81 mm, rescaled from normalised units by y_std=17.838)

in-sample (optimistic, do NOT report this):
  MAE   9.68 mm   RMSE  12.07 mm
  mean sd, mean-of-mix question       4.20 mm
  mean sd, single-reading question   13.50 mm
  ratio 3.21x -- this is the gap the old
  optimiser collapsed by ADDING the noise term instead of removing it


## 6. Gates

**G1 — water direction.** Holds by construction: water reaches the prediction only through
`is_slump_baseline`, whose slope is `[IS]` fixed and positive. The cell below proves it
numerically anyway, because a proof that is also a test is worth more than either alone.

**G5 — leave-one-group-out bias inside ±25 mm.** The band engineers actually order against.

**G6 — out-of-group skill above zero.** Skill below zero means the model is worse than
predicting the overall mean, which is what the previous model scored (−0.28).

In [12]:
# EXPORT: skip -- gates
print("G1  water direction")
print("    A sweep of predicted slump against water would be TAUTOLOGICAL: the GP never")
print("    sees water, so its output is constant across the sweep and the test would")
print("    check only that an affine function increases. Test the real condition instead --")
print("    that no water-derived feature can reach the GP, and that the GP truly ignores water.")

leaked = sorted(set(model["features"]) & WATER_DERIVED)
print(f"    water-derived features in the model : {leaked or 'none'}")
assert not leaked, f"G1 violated -- {leaked} reached the residual model"

# the GP's own output must be INVARIANT to water; if it is not, water leaked in by
# some other name and the baseline is no longer the only route.
probe = pd.concat([kept.iloc[[0]].copy() for _ in range(40)], ignore_index=True)
probe[WATER] = np.linspace(140.0, 200.0, 40)
X = model["scaler"].transform(probe[model["features"]].to_numpy(float))
gp_only = model["gp"].predict(X)
print(f"    distinct GP residual values across 140-200 kg water : {len(np.unique(np.round(gp_only,9)))}")
assert len(np.unique(np.round(gp_only, 9))) == 1, \
    "G1 violated -- the GP output moved with water, so water reached it"
print(f"    so the water slope is exactly [IS] cl 5.3: {is_slump_slope(20):.3f} mm/kg, "
      f"and cannot be anything else")

# and the honest caveat the old sweep concealed
print()
print("    NOTE the baseline is affine and unbounded, and it is NOT a slump prediction.")
print("    At a realistic dose it is sensible; at zero reduction it is not, which is why")
print("    the reduction is a required argument rather than a default:")
for w in (155.0, 165.0, 175.0):
    r0 = float(water_reduction_pct(1.0, WFIT))
    print(f"      water {w:6.1f} kg at {r0:5.2f}% reduction -> baseline "
          f"{float(is_slump_baseline(w, 20, water_reduction_pct=r0)):7.1f} mm"
          f"   |  at 0% -> {float(is_slump_baseline(w, 20)):8.1f} mm")
print("    Monotonicity says nothing about magnitude, and magnitude is what the")
print("    optimiser acts on. That is what the achievability check is for.")

print("\nG5 / G6  leave-one-group-out")
groups = sorted(kept[GRP].unique())
rows, all_resid, resid_by_group = [], [], {}
for g in groups:
    tr, te = kept[kept[GRP] != g], kept[kept[GRP] == g]
    if len(te) < 2 or tr[GRP].nunique() < 2:
        continue
    # [PC] the cl 5.3 water fit is built from MEASURED slump, so it must be refit on the
    #      training fold only. Reusing the all-family WFIT here let the held-out family
    #      help build its own baseline -- that was leakage, and it flattered every number.
    m = fit_residual_model(tr, water_fit=fit_water_reduction(tr))
    mu, sd = predict_slump(m, te, kind="mean")
    truth = te[SLUMP].to_numpy(float)
    bias = float(np.mean(mu - truth))
    mae = float(np.mean(np.abs(mu - truth)))
    base_pred = float(tr[SLUMP].mean())      # the trivial predictor available at predict time
    ss_res = float(np.sum((truth - mu) ** 2))
    ss_base = float(np.sum((truth - base_pred) ** 2))
    skill = 1.0 - ss_res / ss_base if ss_base > 0 else np.nan
    all_resid.extend((mu - truth).tolist())
    resid_by_group[g] = (mu - truth)
    rows.append({"group": g, "n": len(te), "bias_mm": bias, "mae_mm": mae,
                 "skill": skill, "ss_res": ss_res, "ss_base": ss_base,
                 "mae_equals_bias": bool(abs(mae - abs(bias)) < 1e-9)})

logo = pd.DataFrame(rows)
print(logo.to_string(index=False, float_format=lambda x: f"{x:8.2f}"))

if len(logo):
    ERR = logo_error(all_resid, logo["bias_mm"])
    SKILL = pooled_skill(logo)
    print(f"\n  POOLED skill  {SKILL:+7.3f}      <- report this, not the mean of ratios")
    print(f"  mean of per-group ratios {logo['skill'].mean():+7.3f}  (dominated by the worst"
          f" fold; group SS_base spans {logo['ss_base'].min():.0f}-{logo['ss_base'].max():.0f})")
    print()
    print(f"  out-of-group RMSE       {ERR['rmse_mm']:7.2f} mm   <- the WIDTH")
    print(f"  out-of-group sd         {ERR['sd_mm']:7.2f} mm")
    print(f"  RMS of per-family bias  {ERR['bias_rms_mm']:7.2f} mm   <- how far the CENTRE moves")
    print(f"  worst family bias       {ERR['bias_max_mm']:7.2f} mm")
    n_off = int(logo["mae_equals_bias"].sum())
    print(f"  families where MAE == |bias| exactly: {n_off} of {len(logo)}"
          f"   (every held-out row errs the same way -> pure offset, not scatter)")
    print()
    # coverage of the nominal 90% band, per family -- the check that pooling hides
    print("  coverage of the nominal 90% interval, per held-out family:")
    cov_rows = []
    for g in logo["group"]:
        tr, te = kept[kept[GRP] != g], kept[kept[GRP] == g]
        m = fit_residual_model(tr, water_fit=fit_water_reduction(tr))
        mu, sd_in = predict_slump(m, te, kind="mean")
        # [PC] the width floor comes from the OTHER families' residuals only -- scoring
        #      coverage with a width calibrated on the same family is calibration-on-test
        other = np.concatenate([resid_by_group[h] for h in resid_by_group if h != g])
        sd_use = np.maximum(sd_in, float(np.sqrt(np.mean(other ** 2))))
        z = 1.6448536269514722          # [S] two-sided 90%
        inside = ((te[SLUMP].to_numpy(float) >= mu - z * sd_use) &
                  (te[SLUMP].to_numpy(float) <= mu + z * sd_use))
        cov_rows.append({"group": g, "n": len(te), "coverage_pct": 100.0 * inside.mean()})
        print(f"     {g:16s} n={len(te):3d}   {100*inside.mean():6.1f} %")
    cov = pd.DataFrame(cov_rows)
    print(f"     {'POOLED':16s} n={int(cov['n'].sum()):3d}   "
          f"{(cov['coverage_pct']*cov['n']).sum()/cov['n'].sum():6.1f} %"
          f"   <- pooling hides the worst family")
    print()
    print(f"  G5  |bias| <= 25 mm for EVERY held-out family : "
          f"{'PASS' if ERR['bias_max_mm'] <= 25 else 'FAIL'}")
    print(f"  G6  POOLED skill > 0                         : "
          f"{'PASS' if SKILL > 0 else 'FAIL'}")
    print(f"  G9  every family covers >= 80% of its own 90% band : "
          f"{'PASS' if cov['coverage_pct'].min() >= 80 else 'FAIL'}"
          f"   (worst {cov['coverage_pct'].min():.1f}%)")
    print()
    print(f"  Held-out families: {len(logo)}. No interval is quoted on five ratio")
    print(f"  statistics -- a bootstrap over five numbers is decoration, not evidence.")

G1  water direction
    A sweep of predicted slump against water would be TAUTOLOGICAL: the GP never
    sees water, so its output is constant across the sweep and the test would
    check only that an affine function increases. Test the real condition instead --
    that no water-derived feature can reach the GP, and that the GP truly ignores water.
    water-derived features in the model : none
    distinct GP residual values across 140-200 kg water : 1
    so the water slope is exactly [IS] cl 5.3: 4.480 mm/kg, and cannot be anything else

    NOTE the baseline is affine and unbounded, and it is NOT a slump prediction.
    At a realistic dose it is sensible; at zero reduction it is not, which is why
    the reduction is a required argument rather than a default:
      water  155.0 kg at 17.75% reduction -> baseline    60.9 mm   |  at 0% ->    -88.9 mm
      water  165.0 kg at 17.75% reduction -> baseline   115.4 mm   |  at 0% ->    -44.1 mm
      water  175.0 kg at 17.75% reduction 

## 7. Achievability — the honest answer when the band cannot be held

`[PC]` If the model's own uncertainty is wider than the band the customer asked for, no
amount of optimisation can guarantee that band. The system says so, in plain words, before
anything is proposed. That converts the project's most awkward limitation into a feature:
it returns a reason instead of an empty list.

In [13]:
def achievable(model, df, target_slump_mm, tolerance_mm=25.0, confidence=0.90,
               *, out_of_group_sd_mm, out_of_group_bias_mm=0.0,
               water_col="water_kg_m3", msa_col="msa"):
    """[PC] Can this slump band be held, at this confidence, for a NEW mix?

    out_of_group_sd_mm is REQUIRED and has no default. It is the honest error scale,
    measured by leave-one-group-out in this notebook. The check uses the LARGER of it
    and the GP's own sd, because a model cannot be more certain about a new mix than
    it was about a held-out family.

    tolerance_mm default 25 mm: [IS] IS 456 cl 7.1 and IS 4926 give the acceptance
    bands for workability, and is_rules.slump_acceptance_band is the authority once a
    specified value is in hand. 25 mm is only this caller's default.

    Returns a verdict a person can read, not just a boolean.
    """
    from scipy.stats import norm
    mu, sd_in = predict_slump(model, df, kind="mean")
    sd = np.maximum(sd_in, float(out_of_group_sd_mm))
    z = norm.ppf(0.5 + confidence / 2.0)
    half_width = z * sd

    # Two separate ways to miss, and they need separate verdicts:
    #   OFFSET  -- the predicted mean is already outside the band
    #   WIDTH   -- the band is narrower than the model's own uncertainty
    # The earlier version tested only WIDTH, so a mix predicted 156 mm off target was
    # reported as a 38 mm uncertainty problem. Bias is not variance.
    mu = np.atleast_1d(mu)
    offset = np.abs(mu - float(target_slump_mm))
    offset_ok = offset <= tolerance_mm
    # an unseen family's centre can itself sit this far out
    offset_ok_after_bias = (offset + float(out_of_group_bias_mm)) <= tolerance_mm
    width_ok = np.atleast_1d(half_width) <= tolerance_mm
    can = offset_ok_after_bias & width_ok

    verdict = []
    for o, ook, obk, w, h in zip(offset, offset_ok, offset_ok_after_bias,
                                 width_ok, np.atleast_1d(half_width)):
        if not ook:
            verdict.append(
                f"NOT achievable -- OFFSET: the predicted mean is {o:.0f} mm from the "
                f"{target_slump_mm:.0f} mm target, outside the +/-{tolerance_mm:.0f} mm band "
                f"before uncertainty is even considered. This is bias, not variance.")
        elif not obk:
            verdict.append(
                f"NOT achievable -- OFFSET plus family drift: predicted mean is {o:.0f} mm "
                f"out, and an unseen family's centre moves a further "
                f"{out_of_group_bias_mm:.0f} mm on this data.")
        elif not w:
            verdict.append(
                f"NOT achievable -- WIDTH: the mean is on target but a +/-{tolerance_mm:.0f} mm "
                f"band cannot be guaranteed at {confidence:.0%}; the honest half-width is "
                f"{h:.0f} mm.")
        else:
            verdict.append("achievable")

    return {
        "target_slump_mm": target_slump_mm,
        "tolerance_mm": tolerance_mm,
        "confidence": confidence,
        "predicted_mean_mm": mu,
        "offset_from_target_mm": offset,
        "model_sd_in_sample_mm": sd_in,
        "out_of_group_sd_mm": float(out_of_group_sd_mm),
        "out_of_group_bias_mm": float(out_of_group_bias_mm),
        "sd_used_mm": sd,
        "required_half_width_mm": half_width,
        "offset_ok": offset_ok,
        "width_ok": width_ok,
        "achievable": can,
        "verdict": verdict,
    }

In [14]:
# EXPORT: skip -- demonstration
OOG_SD = ERR["rmse_mm"]
OOG_BIAS = ERR["bias_rms_mm"]
print(f"in-sample GP sd        ~{predict_slump(model, kept, kind='mean')[1].mean():5.2f} mm")
print(f"out-of-group RMSE       {OOG_SD:5.2f} mm   <- the number achievability uses")
print(f"ratio                   {OOG_SD/max(predict_slump(model, kept, kind='mean')[1].mean(),1e-9):5.1f}x")
print()
probe = kept.iloc[:5].copy()
a = achievable(model, probe, target_slump_mm=120.0, tolerance_mm=25.0,
               out_of_group_sd_mm=OOG_SD, out_of_group_bias_mm=OOG_BIAS)
for i, (mu, hw, ok, v) in enumerate(zip(a["predicted_mean_mm"],
                                        a["required_half_width_mm"],
                                        a["achievable"], a["verdict"])):
    print(f"  row {i}: predicted {mu:6.1f} mm, 90% half-width {hw:5.1f} mm -> {v}")
print()
print("This is the system telling the truth instead of returning an empty list.")

in-sample GP sd        ~ 4.20 mm
out-of-group RMSE       19.43 mm   <- the number achievability uses
ratio                     4.6x

  row 0: predicted  125.2 mm, 90% half-width  32.0 mm -> NOT achievable -- WIDTH: the mean is on target but a +/-25 mm band cannot be guaranteed at 90%; the honest half-width is 32 mm.
  row 1: predicted   93.3 mm, 90% half-width  32.0 mm -> NOT achievable -- OFFSET: the predicted mean is 27 mm from the 120 mm target, outside the +/-25 mm band before uncertainty is even considered. This is bias, not variance.
  row 2: predicted   96.1 mm, 90% half-width  32.0 mm -> NOT achievable -- OFFSET plus family drift: predicted mean is 24 mm out, and an unseen family's centre moves a further 11 mm on this data.
  row 3: predicted  122.3 mm, 90% half-width  32.0 mm -> NOT achievable -- WIDTH: the mean is on target but a +/-25 mm band cannot be guaranteed at 90%; the honest half-width is 32 mm.
  row 4: predicted   99.6 mm, 90% half-width  32.0 mm -> NOT achievable -

## 8. Save

In [15]:
# EXPORT: skip -- artefacts
model["out_of_group_sd_mm"] = OOG_SD
model["out_of_group_bias_mm"] = OOG_BIAS
model["logo_error"] = ERR
model["pooled_skill"] = SKILL
model["coverage"] = cov.to_dict("records")
model["water_reduction_fit"] = WFIT
model["logo"] = logo.to_dict("records")
joblib.dump(model, "workability_residual_model.joblib")
Path("workability_quarantine.json").write_text(json.dumps(qrep, indent=2), encoding="utf-8")
if len(logo):
    logo.to_csv("workability_logo_report.csv", index=False)
print("wrote workability_residual_model.joblib")
print("wrote workability_quarantine.json")
print("wrote workability_logo_report.csv")
print()
print("NOTE: this model supersedes workability_model.joblib, which is shared byte-for-byte")
print("with the FinalSDL pipeline and is the one that points the wrong way on water.")
print("Do not load the old file in notebooks 11-13.")

wrote workability_residual_model.joblib
wrote workability_quarantine.json
wrote workability_logo_report.csv

NOTE: this model supersedes workability_model.joblib, which is shared byte-for-byte
with the FinalSDL pipeline and is the one that points the wrong way on water.
Do not load the old file in notebooks 11-13.


## Export

In [16]:
"""Emit workability_residual.py from this notebook's own cells.

Same mechanism as 00_is_codes_layer.ipynb: the module is a concatenation of the
cells that were just executed, so what ships is exactly what was tested. Fixture
and export cells are excluded -- fixtures stay in the notebook where a human runs
them and reads the evidence.
"""
import json as _json
from pathlib import Path as _Path

NOTEBOOK = "10_workability_residual_GP.ipynb"
MODULE   = "workability_residual.py"

_Q = chr(34) * 3
_HEADER = "\n".join([
    _Q + "Workability - IS 10262 baseline plus a residual GP.",
    "",
    "GENERATED FILE - do not edit. Edit {nb} and re-run its export cell.",
    "",
    "Water enters the slump prediction ONLY through is_slump_baseline(), whose slope",
    "is fixed by IS 10262:2019 Table 4 + cl 5.3. The GP never sees a water-derived",
    "feature, so the water direction cannot be learned wrong (gate G1).",
    "",
    "predict_slump(kind='mean') REMOVES the WhiteKernel term; kind='specimen' keeps it.",
    _Q,
    "from __future__ import annotations",
    "",
    "import numpy as np",
    "import pandas as pd",
    "",
    "import is_rules as IS",
    "", "",
])


def export(notebook: str = NOTEBOOK, path: str = MODULE) -> None:
    nb = _json.loads(_Path(notebook).read_text(encoding="utf-8"))
    out, skipped = [], 0
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if ("def export(" in src or "# EXPORT: skip" in src
                or src.lstrip().startswith("def _t(") or src.lstrip().startswith("!")):
            skipped += 1
            continue
        lines = [ln for ln in src.split("\n")
                 if not ln.startswith(("from __future__", "import ", "from dataclasses",
                                       "from enum import", "from pathlib", "from typing",
                                       "import numpy", "import pandas"))
]
        body = "\n".join(lines).strip("\n")
        if body:
            out.append(body)

    _Path(path).write_text(_HEADER.format(nb=notebook) + "\n\n\n".join(out) + "\n",
                           encoding="utf-8")
    ns: dict = {}
    exec(compile(_Path(path).read_text(encoding="utf-8"), path, "exec"), ns)
    assert "is_slump_baseline" in ns and "predict_slump" in ns
    assert "inside_fitted_range" in ns
    assert "out_of_group_sd_mm" in ns["achievable"].__code__.co_varnames
    assert "water_reduction_pct" in ns and "fit_water_reduction" in ns
    assert "pooled_skill" in ns and "logo_error" in ns
    assert ns["WATER_DERIVED"] & set(ns["RESIDUAL_FEATURES"]) == set()
    assert ns["is_slump_baseline"](200.0, 20) > ns["is_slump_baseline"](190.0, 20)
    print(f"wrote {path}: {len(out)} cells merged, {skipped} skipped, import-checked OK")


export()

wrote workability_residual.py: 5 cells merged, 11 skipped, import-checked OK


## Open items — carried, not resolved

1. `[U]` The fine-fraction flag uses the single most permissive cap the standard allows
   (Zone I at 20 mm with cl 5.5.2). The per-row cap needs each row's own zone and w/c,
   because `[IS]` cl 5.5.1 moves it. The flagged count may rise once that is applied.
2. `[EI]` Inverting cl 5.3 into a response slope is this project's interpretation.
   cl 5.3 ends *"or may be established by trial"*, and once trials exist that vary water
   **independently of superplasticiser and binder**, the slope becomes measurable and this
   `[EI]` can be replaced by a `[V]` plant-specific value.
3. `[ML]` The RBF kernel is a default, not a comparison. No claim is made that it beats
   Matérn or a linear model on this data; with five groups such a comparison would not be
   conclusive either way.
4. The residual's intercept absorbs the superplasticiser water reduction that `[IS]`
   cl 5.3 permits (20 to 30 percent and above). If the plant changes admixture brand, the
   intercept is stale and the model must be refit — this is what the `state_hash` on every
   proposal exists to catch.